# UDA-Hub - Agentic Customer Support

Run `01_external_db_setup.ipynb` and `02_core_db_setup.ipynb` first. Part B needs `VOCAREUM_API_KEY` (or `OPENAI_API_KEY`) in `.env` (see `.env.example`).

- **Part A** demonstrates the building blocks without an LLM: data retrieval, RAG with confidence scores, MCP tools, memory, routing.
- **Part B** runs complete tickets through the multi-agent graph (classification → routing → retrieval/tools → resolution or escalation → persistence).

Design: `agentic/design/README.md`. Tests: `pytest`.

## Setup

In [1]:
from dotenv import load_dotenv
from utils import chat_interface

In [2]:
load_dotenv()

True

In [3]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

from agentic.workflow import orchestrator

## Part A - building blocks (no LLM needed)

### A1. The multi-agent graph

In [4]:
print(orchestrator.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	ticket_intake(ticket_intake)
	classifier(classifier)
	supervisor(supervisor)
	knowledge_agent(knowledge_agent)
	account_agent(account_agent)
	account_tools(account_tools)
	escalation_agent(escalation_agent)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> ticket_intake;
	account_agent -.-> account_tools;
	account_agent -.-> escalation_agent;
	account_agent -.-> finalize;
	account_tools -.-> account_agent;
	account_tools -.-> escalation_agent;
	classifier --> supervisor;
	escalation_agent --> finalize;
	knowledge_agent -.-> escalation_agent;
	knowledge_agent -.-> finalize;
	supervisor -.-> account_agent;
	supervisor -.-> escalation_agent;
	supervisor -.-> knowledge_agent;
	ticket_intake --> classifier;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### A2. Data retrieval from the databases

In [5]:
from agentic.tools.db import session_scope, udahub_engine, cultpass_engine
from data.models import udahub, cultpass

with session_scope(udahub_engine()) as s:
    for model in [udahub.Account, udahub.User, udahub.Ticket, udahub.TicketMetadata,
                  udahub.TicketMessage, udahub.Knowledge, udahub.CustomerMemory]:
        print(f"udahub.{model.__tablename__:18s} rows={s.query(model).count()}")
with session_scope(cultpass_engine()) as s:
    for model in [cultpass.User, cultpass.Subscription, cultpass.Experience, cultpass.Reservation]:
        print(f"cultpass.{model.__tablename__:16s} rows={s.query(model).count()}")

udahub.accounts           rows=1
udahub.users              rows=2
udahub.tickets            rows=2
udahub.ticket_metadata    rows=2
udahub.ticket_messages    rows=3
udahub.knowledge          rows=16
udahub.customer_memories  rows=2
cultpass.users            rows=6
cultpass.subscriptions    rows=6
cultpass.experiences      rows=7
cultpass.reservations     rows=6


### A3. RAG: retrieval for different ticket types, with the relevance score used to decide escalation
The knowledge agent escalates when the best score is below `MIN_RELEVANCE`; otherwise the LLM answers only from these articles and returns its own confidence (escalates below 0.5).

In [6]:
from agentic.agents.knowledge import MIN_RELEVANCE
from agentic.tools.mcp_client import call_tool

tickets = {
    "technical": "The QR code is not showing in the app",
    "login": "I forgot my password and can't sign in",
    "billing": "My card payment failed this month",
    "account": "How do I change the email on my account?",
    "policy": "Can I cancel my reservation the day before?",
    "accessibility": "Is there wheelchair access at events?",
    "off-topic": "Can you recommend a good pizza recipe?",
}
for kind, text in tickets.items():
    hit = call_tool("udahub", "search_knowledge", {"query": text, "k": 1})["results"][0]
    decision = "answer from KB" if hit["score"] >= MIN_RELEVANCE else "ESCALATE (no relevant article)"
    print(f"{kind:13s} score={hit['score']:.3f}  {decision:31s} -> {hit['title']}")

technical     score=0.590  answer from KB                  -> App Troubleshooting: Crashes and QR Code Not Showing


login         score=0.399  answer from KB                  -> How to Handle Login Issues?


billing       score=0.537  answer from KB                  -> Updating Payment Method and Failed Payments


account       score=0.584  answer from KB                  -> Changing Account Email or Personal Details


policy        score=0.483  answer from KB                  -> How to Cancel a Reservation (24-Hour Policy)


accessibility score=0.537  answer from KB                  -> Accessibility and Special Assistance


off-topic     score=0.102  ESCALATE (no relevant article)  -> Updating Payment Method and Failed Payments


### A4. Support-operation tools over MCP (structured responses, validation and business rules)
These are the same calls the account agent makes. A failed business rule returns `ok: false` plus a reason, and never raises.

In [7]:
from agentic.tools.mcp_client import list_tools
print("cultpass MCP tools:", list_tools("cultpass"))
print("udahub MCP tools:  ", list_tools("udahub"))
profile = call_tool("cultpass", "get_customer_profile", {"user_id": "f556c0"})
print("\nprofile:", {k: profile[k] for k in ("full_name", "is_blocked")}, profile["subscription"]["tier"], "remaining:", profile["subscription"]["remaining_quota"])
samba = call_tool("cultpass", "search_experiences", {"query": "samba"})["experiences"][0]
print("search:", samba["experience_id"], samba["title"], samba["when"], "slots:", samba["slots_available"])

cultpass MCP tools: ['get_customer_profile', 'list_reservations', 'search_experiences', 'reserve_experience', 'cancel_reservation', 'pause_subscription', 'resume_subscription', 'cancel_subscription', 'change_subscription_tier', 'request_refund', 'list_refund_requests', 'update_profile', 'request_email_change', 'find_customer', 'decide_refund', 'set_account_block']
udahub MCP tools:   ['search_knowledge', 'get_ticket', 'create_ticket', 'add_ticket_message', 'update_ticket_metadata', 'get_customer_ticket_history', 'save_memory', 'search_memories', 'list_tickets', 'assign_ticket']

profile: {'full_name': 'Bob Stone', 'is_blocked': False} premium remaining: 6
search: 05a503 Samba Night at Lapa 2026-11-11 22:47 slots: 23


In [8]:
# successful action + undo
booked = call_tool("cultpass", "reserve_experience", {"user_id": "f556c0", "experience_id": samba["experience_id"]})
print("reserve:", booked)
print("cancel :", call_tool("cultpass", "cancel_reservation", {"user_id": "f556c0", "reservation_id": booked["reservation_id"]}))

# validation / business-rule errors
premium = next(e for e in call_tool("cultpass", "search_experiences", {})["experiences"] if e["is_premium"])
print("blocked user  :", call_tool("cultpass", "reserve_experience", {"user_id": "a4ab87", "experience_id": samba["experience_id"]})["error"])
print("cancelled sub :", call_tool("cultpass", "reserve_experience", {"user_id": "f1f10d", "experience_id": samba["experience_id"]})["error"])
print("basic->premium:", call_tool("cultpass", "reserve_experience", {"user_id": "88382b", "experience_id": premium["experience_id"]})["error"])
print("unknown event :", call_tool("cultpass", "reserve_experience", {"user_id": "f556c0", "experience_id": "zzz"})["error"])
print("bad arguments :", call_tool("cultpass", "reserve_experience", {"user_id": "f556c0"}))

reserve: {'ok': True, 'reservation_id': '7d09e1', 'experience_id': '05a503', 'title': 'Samba Night at Lapa', 'location': 'Rio de Janeiro, Brazil', 'when': '2026-11-11 22:47', 'remaining_quota': 5, 'message': "Reserved 'Samba Night at Lapa'. Arrive 15 minutes early with your QR code."}
cancel : {'ok': True, 'reservation_id': '7d09e1', 'title': 'Samba Night at Lapa', 'message': "Reservation for 'Samba Night at Lapa' cancelled; the credit was returned to your monthly quota."}
blocked user  : account_blocked
cancelled sub : subscription_inactive
basic->premium: premium_requires_upgrade
unknown event : experience_not_found
bad arguments : {'ok': False, 'error': 'tool_error', 'message': "1 validation error for call[reserve_experience]\nexperience_id\n  Missing required argument [type=missing_argument, input_value={'user_id': 'f556c0'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.14/v/missing_argument"}


### A4b. Subscription management and refunds (two-step confirmation, validation)
Changes need `confirm=true`. Refunds are only ever filed as `pending_approval`; a human approves them with a staff tool.

In [9]:
print("pause preview :", call_tool("cultpass", "pause_subscription", {"user_id": "88382b"}))
print("bad tier      :", call_tool("cultpass", "change_subscription_tier", {"user_id": "88382b", "new_tier": "gold"})["error"])
print("refund (bad)  :", call_tool("cultpass", "request_refund", {"user_id": "88382b", "reason": "x"})["error"])
print("email (taken) :", call_tool("cultpass", "request_email_change", {"user_id": "88382b", "new_email": "bob.stone@granite.com"})["error"])

pause preview : {'ok': True, 'requires_confirmation': True, 'action': 'pause_subscription', 'preview': 'The subscription will be paused immediately; existing reservations stay valid.', 'message': 'The subscription will be paused immediately; existing reservations stay valid. Ask the customer to confirm, then call again with confirm=true.'}
bad tier      : invalid_tier
refund (bad)  : invalid_reason
email (taken) : email_in_use


### A5. Long-term memory and history for a returning customer (Bob Stone, `f556c0`)

In [10]:
print(call_tool("udahub", "get_customer_ticket_history", {"account_id": "cultpass", "external_user_id": "f556c0"}))
for m in call_tool("udahub", "search_memories", {"account_id": "cultpass", "external_user_id": "f556c0", "query": "music events"})["memories"]:
    print(m["memory_type"], "|", m["content"])

{'ok': True, 'tickets': [{'ticket_id': 'past-ticket-bob-001', 'created_at': '2026-10-18 22:47:41.872992', 'status': 'resolved', 'issue_type': 'experience_info', 'first_message': 'Are premium experiences included in my plan? I love the samba events in Rio.'}]}
preference | Enjoys samba / music events in Rio de Janeiro
interaction | 2026-10-18 [experience_info] Asked whether premium experiences are included -> resolved: included in Premium tier.


### A6. Routing decisions on sample classified tickets
The inputs are the classifier's output (category, urgency, sentiment, confidence) plus ticket metadata (status, turn count) and customer context (whether the account is blocked).

In [11]:
from agentic.agents.supervisor import compute_priority, decide_route

samples = [
    ("How do I reset my password?",          {"category": "login_access", "urgency": "medium", "sentiment": "neutral"}, {}),
    ("I can't log in!",                       {"category": "login_access", "urgency": "medium", "sentiment": "neutral"}, {"customer_context": {"is_blocked": True}}),
    ("Book Samba Night for me",               {"category": "reservation",  "urgency": "low",    "sentiment": "positive"}, {}),
    ("I was charged twice",                   {"category": "billing_refund", "urgency": "high", "sentiment": "negative"}, {"external_user_id": "88382b"}),
    ("App crashes, my event starts in 1h!!",  {"category": "technical_issue", "urgency": "high", "sentiment": "negative"}, {}),
    ("Still can't pause my plan...",          {"category": "subscription", "urgency": "medium", "sentiment": "neutral"}, {"turn": 6}),
    ("Hmm, something about stuff",            {"category": "general_question", "confidence": 0.2}, {}),
]
for text, cls, extra in samples:
    state = {"classification": cls, **extra}
    route, reason = decide_route(state)
    print(f"{text:38s} -> {route:17s} {compute_priority(state)[0]}  ({reason})")

How do I reset my password?            -> knowledge_agent   P3  ('login_access' can be answered from the knowledge base)
I can't log in!                        -> escalation_agent  P3  (policy: blocked accounts can only be reviewed by trust & safety (AI cannot unblock))
Book Samba Night for me                -> account_agent     P4  ('reservation' needs customer data or an account action)
I was charged twice                    -> account_agent     P1  (policy: file a refund request, then hand it to the billing team)
App crashes, my event starts in 1h!!   -> knowledge_agent   P2  ('technical_issue' can be answered from the knowledge base)
Still can't pause my plan...           -> escalation_agent  P3  (complexity: still unresolved after 5 AI turns)
Hmm, something about stuff             -> knowledge_agent   P3  (low classifier confidence: try the knowledge base (escalates if nothing relevant))


## Part B - end-to-end ticket processing with the LLM
Each scenario uses its own ticket (`thread_id` == ticket id). `external_user_id` identifies the CultPass customer on the first turn; `ticket_metadata` carries platform metadata such as the channel.

In [12]:
from langchain_core.messages import HumanMessage

def ask(ticket_id, text, external_user_id=None, payload=None, **configurable):
    payload = payload or {"messages": [HumanMessage(text)]}
    if external_user_id:
        payload["external_user_id"] = external_user_id
    state = orchestrator.invoke(payload, config={"configurable": {"thread_id": ticket_id, **configurable}})
    reply = state["messages"][-1]
    cls = state.get("classification") or {}
    print(f"User: {payload['messages'][0].content}\nAssistant ({reply.name}): {reply.content}")
    print(f"  -> category={cls.get('category')} sentiment={cls.get('sentiment')} priority={state.get('priority')} "
          f"route={state['route']} status={state['ticket_status']}\n     reason: {state.get('route_reason')}\n")
    return state

**B1. Resolved from the knowledge base (hybrid semantic RAG)**: a paraphrased question with no keyword overlap

In [13]:
state = ask("nb-ticket-1", "When I open my ticket in the app there is no code to scan at the door", external_user_id="88382b")
[(a["title"], a["score"]) for a in state["retrieved_articles"]]

User: When I open my ticket in the app there is no code to scan at the door
Assistant (knowledge_agent): Please update the app and log out and back in - that refreshes your QR codes. If it still doesn't appear, venue staff can check you in with your email.
  -> category=technical_issue sentiment=negative priority=P2 route=knowledge_agent status=resolved
     reason: 'technical_issue' can be answered from the knowledge base



[('App Troubleshooting: Crashes and QR Code Not Showing', 0.5251),
 ('How to Reserve a Spot for an Event', 0.3171),
 ('Bringing a Guest to an Experience', 0.3056)]

**B2. Tool usage and account actions over several turns (short-term memory)**: the second turn doesn't repeat the customer ID

In [14]:
_ = ask("nb-ticket-2", "Which reservations do I have?", external_user_id="f556c0")
_ = ask("nb-ticket-2", "Please also book Samba Night at Lapa for me")

User: Which reservations do I have?
Assistant (account_agent): You have two reservations:

1. **Ibirapuera Park Bike Ride**  
   - **Date:** November 14, 2026  
   - **Location:** São Paulo, Brazil  
   - **Reservation ID:** 579dc3  
   - **Status:** Reserved  
   - **Premium Experience:** Yes  

2. **Modern Art at MASP**  
   - **Date:** November 13, 2026  
   - **Location:** São Paulo, Brazil  
   - **Reservation ID:** 883607  
   - **Status:** Reserved  
   - **Premium Experience:** No
  -> category=reservation sentiment=neutral priority=P3 route=account_agent status=resolved
     reason: 'reservation' needs customer data or an account action



User: Please also book Samba Night at Lapa for me
Assistant (account_agent): I've successfully reserved **Samba Night at Lapa** for you on **November 11, 2026, at 22:47**. Your reservation ID is **64bbe7**. Remember to arrive 15 minutes early with your QR code ready!
  -> category=reservation sentiment=neutral priority=P3 route=account_agent status=resolved
     reason: 'reservation' needs customer data or an account action



**B3. Subscription management with two-step confirmation**

In [15]:
from agentic.tools.cultpass_tools import get_customer_profile
_ = ask("nb-ticket-3", "I'm travelling next month, can you pause my subscription?", external_user_id="88382b")
print("status before confirmation:", get_customer_profile("88382b")["subscription"]["status"])
_ = ask("nb-ticket-3", "Yes, please go ahead")
print("status after confirmation: ", get_customer_profile("88382b")["subscription"]["status"])

User: I'm travelling next month, can you pause my subscription?
Assistant (account_agent): I can pause your subscription immediately, and your existing reservations will remain valid. Please confirm if you'd like me to proceed with this action.
  -> category=subscription sentiment=neutral priority=P4 route=account_agent status=in_progress
     reason: 'subscription' needs customer data or an account action

status before confirmation: active


User: Yes, please go ahead
Assistant (account_agent): Your subscription has been successfully paused. You can resume it anytime. Enjoy your travels, Cathy!
  -> category=subscription sentiment=neutral priority=P4 route=account_agent status=resolved
     reason: a previewed account change is waiting for the customer's confirmation

status after confirmation:  paused


**B4. Refund processing**: the AI files the request, the ticket goes to the billing queue, and a human approves it with the staff MCP tool

In [16]:
state = ask("nb-ticket-4", "I was charged twice this month and want my money back!", external_user_id="f556c0")
refund = call_tool("cultpass", "list_refund_requests", {"user_id": "f556c0"})["refunds"][-1]
print("refund request:", refund["refund_id"], refund["status"])
print("staff decision:", call_tool("cultpass", "decide_refund", {"refund_id": refund["refund_id"], "approve": True,
                                                              "decided_by": "lead@cultpass", "note": "duplicate charge confirmed"}))

User: I was charged twice this month and want my money back!
Assistant (escalation_agent): A human specialist will review your ticket and get back to you via email soon.
  -> category=billing_refund sentiment=angry priority=P1 route=account_agent status=escalated
     reason: policy: file a refund request, then hand it to the billing team

refund request: rf-148e95 pending_approval
staff decision: {'ok': True, 'refund_id': 'rf-148e95', 'status': 'approved'}


**B5. Escalations**: a blocked account (on the existing ticket from notebook 02) and a question with no matching article

In [17]:
from agentic.tools.udahub_tools import get_customer_ticket_history
alice_ticket = get_customer_ticket_history("cultpass", "a4ab87")["tickets"][0]["ticket_id"]
_ = ask(alice_ticket, "I still can't log in. Can you help?")
_ = ask("nb-ticket-5", "Can you recommend a good pizza recipe?", external_user_id="e6376d")

User: I still can't log in. Can you help?
Assistant (escalation_agent): Hi Alice, a human specialist will review your ticket and get back to you via email soon.
  -> category=login_access sentiment=neutral priority=P2 route=escalation_agent status=escalated
     reason: policy: blocked accounts can only be reviewed by trust & safety (AI cannot unblock)



User: Can you recommend a good pizza recipe?
Assistant (escalation_agent): A human specialist will review your request and reply to you via email soon.
  -> category=general_question sentiment=neutral priority=P4 route=knowledge_agent status=escalated
     reason: 'general_question' can be answered from the knowledge base



**B6. Multiple channels**: e-mail (customer identified by the sender address, quoted text and signature stripped, e-mail formatting) and a public social-media post (privacy-safe, short)

In [18]:
from agentic.channels import from_email, from_social
_ = ask("nb-ticket-6", None, payload=from_email(
    "Bob Stone <bob.stone@granite.com>", "Bringing my wife",
    "Hello,\nCan my wife come with me to Samba Night?\n\nThanks,\nBob\n\nOn Mon, CultPass wrote:\n> previous message"))
_ = ask("nb-ticket-7", None, payload=from_social("@samba_fan", "@cultpass the app crashed during checkout and I missed my event tonight!!"))

User: Subject: Bringing my wife

Hello,
Can my wife come with me to Samba Night?
Assistant (account_agent): Hi Bob,

For Samba Night at Lapa on November 11, each reservation is personal and valid for one person. If your wife would like to join you, she will need her own CultPass reservation. 

Many events allow you to add a paid guest ticket through the app. You can check the event page for Samba Night to see if this option is available.

Let me know if you need help with anything else!

Best regards,
CultPass Support (UDA-Hub)
  -> category=reservation sentiment=neutral priority=P3 route=account_agent status=resolved
     reason: 'reservation' needs customer data or an account action



User: the app crashed during checkout and I missed my event tonight!!
Assistant (escalation_agent): A human specialist will review your ticket and respond via email shortly. Please send us a private message for further assistance.
  -> category=technical_issue sentiment=angry priority=P1 route=escalation_agent status=escalated
     reason: sentiment: angry customer with an urgent issue is prioritised to a human



**B7. Long-term memory**: a new ticket for Bob recalls his past tickets and stated preference

In [19]:
state = ask("nb-ticket-8", "Any music events coming up that I would like?", external_user_id="f556c0")
state["long_term_memory"]

User: Any music events coming up that I would like?
Assistant (account_agent): I couldn't find any specific music events in Rio de Janeiro right now, but there's a "Samba Night at Lapa" on November 11, 2026. Would you like to reserve a spot for that?
  -> category=experience_info sentiment=neutral priority=P3 route=account_agent status=resolved
     reason: 'experience_info' needs customer data or an account action



[{'memory_type': 'preference',
  'content': 'Enjoys samba / music events in Rio de Janeiro',
  'ticket_id': 'past-ticket-bob-001',
  'created_at': '2026-10-18 22:47'},
 {'memory_type': 'interaction',
  'content': '2026-11-07 [reservation] Customer is asking if his wife can accompany him to Samba Night. -> resolved by account_agent',
  'ticket_id': 'nb-ticket-6',
  'created_at': '2026-11-07 22:48'},
 {'memory_type': 'interaction',
  'content': '2026-11-07 [billing_refund] Customer was charged twice this month and requests a refund. -> escalated to a human (refund request rf-148e95 filed (pending_approval); needs support-lead decision)',
  'ticket_id': 'nb-ticket-4',
  'created_at': '2026-11-07 22:48'},
 {'memory_type': 'interaction',
  'content': '2026-11-07 [reservation] Customer requests to book Samba Night at Lapa. -> resolved by account_agent; actions: reserve_experience(experience_id=05a503)',
  'ticket_id': 'nb-ticket-2',
  'created_at': '2026-11-07 22:48'},
 {'memory_type': 'inte

**B8. What was persisted**: the staff work queue (priority first) and a ticket's message log

In [20]:
from agentic.tools.udahub_tools import get_ticket
for t in call_tool("udahub", "list_tickets", {"status": "escalated"})["tickets"]:
    print(t["priority"], t["assigned_queue"], t["channel"], t["ticket_id"], t["issue_type"])
print()
for m in get_ticket("nb-ticket-3")["messages"]:
    print(m["role"], "|", m["content"][:110])

P1 technical social nb-ticket-7 technical_issue
P1 billing chat nb-ticket-4 billing_refund
P2 trust_safety chat d0a4e6f9-b239-4571-9988-8370a0015fe2 login_access
P4 general chat nb-ticket-5 general_question

user | I'm travelling next month, can you pause my subscription?
ai | I can pause your subscription immediately, and your existing reservations will remain valid. Please confirm if
user | Yes, please go ahead
ai | Your subscription has been successfully paused. You can resume it anytime. Enjoy your travels, Cathy!


**B9. A/B testing of routing strategies**: with `routing_strategy="ab"` each ticket is assigned to `rules` or `llm` (sticky hash); outcomes are aggregated from the structured log. The offline evaluation on the labelled set is in `agentic/design/ab_test_report.md`.

In [21]:
from agentic.experiments import ab_report
for i, text in enumerate(["How do I change my e-mail address?", "What events are on in Rio?",
                          "My QR code won't load", "How many experiences do I have left?"]):
    ask(f"nb-ab-{i}", text, external_user_id="f556c0", routing_strategy="ab")
ab_report()

User: How do I change my e-mail address?
Assistant (account_agent): To change your email address, you'll need to go to 'My Account' > 'Profile'. A confirmation link will be sent to the new email address for verification. If you want to proceed with a different email, please provide that new email address.
  -> category=account_status sentiment=neutral priority=P3 route=account_agent status=resolved
     reason: llm router: Customer needs assistance with changing their email address on their account.



User: What events are on in Rio?
Assistant (account_agent): In Rio de Janeiro, there are two upcoming events:

1. **Samba Night at Lapa** - Dance the night away at a traditional samba club in the Lapa arches on **November 11, 2026**. There are 22 slots available.

2. **Christ the Redeemer Experience** - Take a guided trip to one of the New Seven Wonders of the World on **November 12, 2026**. This is a premium event with 9 slots available.

Let me know if you'd like to reserve a spot for either event!
  -> category=experience_info sentiment=neutral priority=P3 route=account_agent status=resolved
     reason: 'experience_info' needs customer data or an account action



User: My QR code won't load
Assistant (knowledge_agent): Please update the app and log out and back in - that refreshes your QR codes. If it still doesn't appear, venue staff can check you in with your email.
  -> category=technical_issue sentiment=negative priority=P2 route=knowledge_agent status=resolved
     reason: 'technical_issue' can be answered from the knowledge base



User: How many experiences do I have left?
Assistant (account_agent): You have 5 experiences left this month. Your monthly quota is 8, and you've used 3 so far.
  -> category=general_question sentiment=neutral priority=P3 route=account_agent status=resolved
     reason: llm router: The customer is asking about their remaining experiences, which requires account-specific information.



{'rules': {'tickets': 11,
  'turns': 13,
  'resolution_rate': 0.636,
  'escalation_rate': 0.364,
  'avg_turns_per_ticket': 1.18,
  'tool_calls_per_turn': 0.77,
  'route_mix': {'knowledge_agent': 3,
   'account_agent': 8,
   'escalation_agent': 2}},
 'llm': {'tickets': 2,
  'turns': 2,
  'resolution_rate': 1.0,
  'escalation_rate': 0.0,
  'avg_turns_per_ticket': 1.0,
  'tool_calls_per_turn': 1.5,
  'route_mix': {'account_agent': 2}}}

## Interactive chat
Set `INTERACTIVE = True` to type yourself. Otherwise a short scripted session is replayed through the same `chat_interface`.

In [22]:
INTERACTIVE = False
if INTERACTIVE:
    chat_interface(orchestrator, "1", external_user_id="f556c0")
else:
    import builtins
    script = iter(["What's included in my subscription?", "How many experiences do I have left this month?",
                   "Thanks, that's all!", "quit"])
    real_input, builtins.input = builtins.input, (lambda prompt="": next(script))
    try:
        chat_interface(orchestrator, "1", external_user_id="f556c0")
    finally:
        builtins.input = real_input

UDA-Hub support chat | ticket=1 | customer=f556c0
User: What's included in my subscription?


Assistant (account_agent): Your CultPass subscription includes 4 curated cultural experiences each month. You can choose from art exhibitions, museum entries, music concerts, film screenings, and more. Some premium experiences may require an additional fee, which will be shown during reservation.
   [category=subscription | route=account_agent | ticket_status=resolved]
User: How many experiences do I have left this month?


Assistant (account_agent): You have 5 experiences left this month. Your monthly quota is 8, and you've used 3 so far.
   [category=subscription | route=account_agent | ticket_status=resolved]
User: Thanks, that's all!


Assistant (knowledge_agent): You're welcome, Bob! I'm glad I could help. I'll mark this ticket as resolved - just write again if anything else comes up.
   [category=closing | route=knowledge_agent | ticket_status=resolved]
User: quit
Assistant: Goodbye!


## Inspect state, tool usage and structured logs

In [23]:
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "1",
        }
    }
))[0].values["messages"]

[HumanMessage(content="What's included in my subscription?", additional_kwargs={}, response_metadata={}, id='381a3c3c-ffba-4ec4-a19c-e497c857dc93'),
 AIMessage(content='Your CultPass subscription includes 4 curated cultural experiences each month. You can choose from art exhibitions, museum entries, music concerts, film screenings, and more. Some premium experiences may require an additional fee, which will be shown during reservation.', additional_kwargs={}, response_metadata={}, name='account_agent', id='0bdab05b-ace9-4d10-a172-9fc4540ed203', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='How many experiences do I have left this month?', additional_kwargs={}, response_metadata={}, id='c5d71105-9d4c-4ecd-89d7-dd99c1ff0377'),
 AIMessage(content="You have 5 experiences left this month. Your monthly quota is 8, and you've used 3 so far.", additional_kwargs={}, response_metadata={}, name='account_agent', id='912ccdba-01b1-4cbd-9612-c24bfd0c677c', tool_calls=[], invalid_tool

In [24]:
# per-thread decision trace: classification, routing (variant, priority), tool calls with outcomes, ticket update
for rec in orchestrator.get_state({"configurable": {"thread_id": "nb-ticket-2"}}).values["trace"]:
    print({k: v for k, v in rec.items() if k in ("event", "agent", "category", "route", "reason", "priority",
                                                  "variant", "tool", "args", "ok", "outcome", "status")})

{'event': 'intake', 'agent': 'ticket_intake'}
{'event': 'classification', 'agent': 'classifier', 'category': 'reservation'}
{'event': 'routing', 'agent': 'supervisor', 'route': 'account_agent', 'reason': "'reservation' needs customer data or an account action", 'category': 'reservation', 'variant': 'rules', 'priority': 'P3'}
{'event': 'tool_decision', 'agent': 'account_agent'}
{'event': 'tool_call', 'agent': 'account_agent', 'tool': 'list_my_reservations', 'args': {}, 'ok': True, 'outcome': ''}
{'event': 'answer', 'agent': 'account_agent'}
{'event': 'ticket_update', 'agent': 'ticket_finalizer', 'status': 'resolved', 'priority': 'P3', 'variant': 'rules'}
{'event': 'intake', 'agent': 'ticket_intake'}
{'event': 'classification', 'agent': 'classifier', 'category': 'reservation'}
{'event': 'routing', 'agent': 'supervisor', 'route': 'account_agent', 'reason': "'reservation' needs customer data or an account action", 'category': 'reservation', 'variant': 'rules', 'priority': 'P3'}
{'event': '

In [25]:
# structured, searchable JSONL log (logs/agent_events.jsonl)
from agentic.logging_utils import read_events
for e in read_events(event="escalation")[-4:]:
    print(e["ticket_id"], e["priority"], e["queue"], "|", e["reason"])

nb-ticket-4 high billing | refund request rf-148e95 filed (pending_approval); needs support-lead decision
d0a4e6f9-b239-4571-9988-8370a0015fe2 high trust_safety | routed by supervisor: policy: blocked accounts can only be reviewed by trust & safety (AI cannot unblock)
nb-ticket-5 low general | No relevant knowledge-base article (best score 0.14 < 0.2)
nb-ticket-7 high technical | routed by supervisor: sentiment: angry customer with an urgent issue is prioritised to a human
